In [ ]:
# # 2) Oracle 엔진 (이건 전에 말한대로 XE 접속정보)
# user = "SQLT01"
# pwd = "oracle"   # 실제 비밀번호
# host = "localhost"   # 일단 이렇게
# port = 1521
# service = "xe"

# conn_str = f"oracle+oracledb://{user}:{pwd}@{host}:{port}/?service_name={service}"
# print(conn_str)   # 확인용: 여기엔 대괄호/소괄호/host 같은 게 나오면 안 됨


# # 3) Oracle에 적재
# dm.to_sql("SDTM_DM", conn_str, if_exists="replace", index=False)
# ae.to_sql("SDTM_AE", conn_str, if_exists="replace", index=False)

In [1]:
import pandas as pd
from sqlalchemy import create_engine

# 1) SDTM XPT/CSV 읽기
dm = pd.read_sas("/workspaces/my_data/sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm/dm.xpt"
                 , format="xport", encoding='cp949') # 또는 read_csv
ae = pd.read_sas("/workspaces/my_data/sdtm-adam-pilot-project/updated-pilot-submission-package/900172/m5/datasets/cdiscpilot01/tabulations/sdtm/ae.xpt"
                 , format="xport", encoding='cp949')



In [9]:
dm.columns

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'SUBJID', 'RFSTDTC', 'RFENDTC',
       'RFXSTDTC', 'RFXENDTC', 'RFICDTC', 'RFPENDTC', 'DTHDTC', 'DTHFL',
       'SITEID', 'AGE', 'AGEU', 'SEX', 'RACE', 'ETHNIC', 'ARMCD', 'ARM',
       'ACTARMCD', 'ACTARM', 'COUNTRY', 'DMDTC', 'DMDY'],
      dtype='object')

In [12]:
# 피험자, 군, 인구통계 컬럼
adsl_cols = [
    "STUDYID", "USUBJID",
    "ARM", "ARMCD",
    "AGE", "AGEU",
    "SEX", "RACE",
    "COUNTRY"
    ] 

# STUDYID, USUBJID 
# 모든 ADaM/SDTM의 공통 키라서, ADSL과 다른 ADaM(ADAE, ADLB 등)을 조인할 때 기본 키로 사용된다.​

# ARM, ARMCD
# 치료군 정보로, 군별 인원수(N), 군별 AE 발생률, 효과 비교 등 거의 모든 테이블의 기본 그룹 변수다.​

# AGE, AGEU
# 나이와 단위(보통 YEARS)로, 고령자/비고령자 같은 서브그룹, 공변량(covariate)로 자주 사용된다.​

# SEX, RACE, COUNTRY
# 성별, 인종, 국가/지역 변수로, 서브그룹 분석과 인구통계 요약(Table 1)에서 필수적인 분류 변수다.

In [14]:
adsl_cols = [c for c in adsl_cols if c in ae.columns]
adae = ae[adsl_cols].copy()

In [16]:
# TRT 변수를 붙이고 싶으면 ADSL에서 ARM 정보 merge
if "ARM" in adsl_cols:
    adae = adae.merge(adsl[["USUBJID", "ARM"]], on="USUBJID", how="left")

adae.head()

,STUDYID,USUBJID
0,CDISCPILOT01,01-701-1015
1,CDISCPILOT01,01-701-1015
2,CDISCPILOT01,01-701-1015
3,CDISCPILOT01,01-701-1023
4,CDISCPILOT01,01-701-1023


In [18]:
# 군별 피험자 수
if "ARM" in adsl_cols:
    n_by_arm = adsl.groupby("ARM")["USUBJID"].nunique()
    print(n_by_arm)

In [19]:
# AE 경험 여부 플래그
adae["AEFL"] = "Y"

# 군별 AE 경험 피험자 수
if "ARM" in adae.columns:
    ae_subj_by_arm = adae.groupby("ARM")["USUBJID"].nunique()
    print(ae_subj_by_arm)


In [20]:
if "ARM" in adae.columns and "AESOC" in adae.columns:
    soc_counts = adae.groupby(["ARM", "AESOC"])["USUBJID"].count().reset_index(name="N_AE")
    soc_counts.head()


In [21]:
dm.columns

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'SUBJID', 'RFSTDTC', 'RFENDTC',
       'RFXSTDTC', 'RFXENDTC', 'RFICDTC', 'RFPENDTC', 'DTHDTC', 'DTHFL',
       'SITEID', 'AGE', 'AGEU', 'SEX', 'RACE', 'ETHNIC', 'ARMCD', 'ARM',
       'ACTARMCD', 'ACTARM', 'COUNTRY', 'DMDTC', 'DMDY'],
      dtype='object')

In [22]:
ae.columns

Index(['STUDYID', 'DOMAIN', 'USUBJID', 'AESEQ', 'AESPID', 'AETERM', 'AELLT',
       'AELLTCD', 'AEDECOD', 'AEPTCD', 'AEHLT', 'AEHLTCD', 'AEHLGT',
       'AEHLGTCD', 'AEBODSYS', 'AEBDSYCD', 'AESOC', 'AESOCCD', 'AESEV',
       'AESER', 'AEACN', 'AEREL', 'AEOUT', 'AESCAN', 'AESCONG', 'AESDISAB',
       'AESDTH', 'AESHOSP', 'AESLIFE', 'AESOD', 'AEDTC', 'AESTDTC', 'AEENDTC',
       'AESTDY', 'AEENDY'],
      dtype='object')